# Swing Trading 1.8 — P0.0 foundation smoke (PRIVATE repository)

**OPTIONAL / ON HOLD — NOT REQUIRED.** Current primary workflow is ChatGPT Python + GitHub Actions CI. No Colab login, token, setup, or run is needed unless the user later specifically requests this spare experiment.

**Engineering fixtures only.** No SSI connection, live signal, broker order, trade simulator, or real backtest is performed here.

1. In Colab, open this notebook using **File → Open notebook → GitHub → Include private repos** and authorize your account.
2. In Colab **Secrets** (key icon), create `GITHUB_READ_TOKEN` as a fine-grained GitHub token with **Contents: Read-only** permission for `ngluonghoa/swing-trading-1-8` only. Allow notebook access to this secret. Never paste the token into cells or save it to GitHub/Drive.
3. Execute the cells in order. The notebook downloads a **commit-pinned** zip snapshot to an ephemeral Colab VM, installs the Python package, and runs synthetic tests. It does not write back to GitHub.

The secret is required for accessing the **private** repository code within the runtime; opening the notebook through Colab's GitHub browser alone does not grant Python code an authenticated git clone. If secret access is missing, the notebook fails clearly.

In [ ]:
import io
import json
import tempfile
import urllib.request
import zipfile
from pathlib import Path, PurePosixPath

from google.colab import userdata

def download_pinned_private_repo():
    token = userdata.get('GITHUB_READ_TOKEN')
    if not token:
        raise RuntimeError('Add Colab secret GITHUB_READ_TOKEN (fine-grained Contents: Read-only) and grant this notebook access')
    api = 'https://api.github.com/repos/ngluonghoa/swing-trading-1-8'
    headers = {'Authorization': 'Bearer ' + token, 'Accept': 'application/vnd.github+json', 'User-Agent': 'swing-trading-foundation-smoke'}
    def get(url):
        req = urllib.request.Request(url, headers=headers)
        with urllib.request.urlopen(req, timeout=60) as response:
            data = response.read(50 * 1024 * 1024 + 1)
        if len(data) > 50 * 1024 * 1024:
            raise RuntimeError('Repository zip exceeds 50 MiB smoke-test limit')
        return data
    sha = json.loads(get(api + '/commits/main').decode('utf-8'))['sha']
    snapshot = get(api + '/zipball/' + sha)
    workspace = Path(tempfile.mkdtemp(prefix='swing_p0_'))
    with zipfile.ZipFile(io.BytesIO(snapshot)) as archive:
        names = [PurePosixPath(info.filename) for info in archive.infolist()]
        if any(name.is_absolute() or '..' in name.parts for name in names):
            raise RuntimeError('Unsafe archive path')
        archive.extractall(workspace)
    candidates = [p for p in workspace.iterdir() if p.is_dir() and (p / 'pyproject.toml').is_file()]
    if len(candidates) != 1:
        raise RuntimeError('Expected exactly one Python repository root in the snapshot')
    return candidates[0], sha

repo_root, pinned_sha = download_pinned_private_repo()
print('Loaded private source snapshot at Git commit:', pinned_sha)


In [ ]:
import subprocess
import sys

subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', str(repo_root)], check=True)
subprocess.run([sys.executable, '-m', 'unittest', 'discover', '-s', 'tests', '-v'], cwd=repo_root, check=True)
print('P0.0 engineering smoke: tests completed, not historical backtest validation')


In [ ]:
from datetime import datetime, timezone
from swing_trading.contracts import DecisionStatus, Strategy
from swing_trading.engine import evaluate_unimplemented_zones

zones = evaluate_unimplemented_zones(Strategy.SHORT, datetime.now(timezone.utc))
assert len(zones) == 16
assert all(zone.status is DecisionStatus.UNVERIFIED for zone in zones)
print('FAIL-CLOSED smoke PASS: 16/16 zones UNVERIFIED. NO REAL SIGNAL / NO TRADE.')
